# Report — Investment Potential Map

Renders the map shown in the README from the gold `revenue_per_postcode` table, so the
picture always matches the pipeline output.

- Colour = `potential_score` computed in `03_gold` (thirds of postcodes by best
  achievable monthly income; 0 = fewer than 5 listings)
- Output: `figures/investment_potential_map.png` in the pipeline volume
  (locally: `docs/images/investment_potential_map.png`)

In [ ]:
# Setup (identical in every pipeline notebook).
# On Databricks, catalog / schema / volume come from the job parameters defined in
# databricks.yml, or from the widgets at the top when run interactively. A blank
# catalog means "the workspace's default catalog". Nothing workspace-specific is
# hardcoded. Locally, all files live under data/output/.
import os
import sys

ON_DATABRICKS = "dbutils" in globals()  # dbutils exists only on Databricks

if ON_DATABRICKS:
    dbutils.widgets.text("catalog", "", "Catalog (blank = workspace default)")
    dbutils.widgets.text("schema", "amsterdam_rental", "Schema")
    dbutils.widgets.text("volume", "pipeline", "Volume")
    dbutils.widgets.text("repo_root", "", "Repo root (blank = auto-detect)")
    _nb_path = (
        dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    )
    _repo = dbutils.widgets.get("repo_root") or "/".join(_nb_path.split("/")[:-2])
    REPO_ROOT = _repo if _repo.startswith("/Workspace/") else "/Workspace" + _repo
else:
    REPO_ROOT = os.path.abspath("..")
sys.path.insert(0, os.path.join(REPO_ROOT, "src"))

from amsterdam_rental.config import PipelineConfig, get_local_spark, resolve_catalog

if ON_DATABRICKS:
    cfg = PipelineConfig.for_databricks(
        REPO_ROOT,
        catalog=resolve_catalog(dbutils.widgets.get("catalog"), spark.catalog.currentCatalog()),
        schema=dbutils.widgets.get("schema"),
        volume=dbutils.widgets.get("volume"),
    )
else:
    spark = get_local_spark("amsterdam-rental-report")
    cfg = PipelineConfig.for_local(REPO_ROOT)
print(cfg.describe())

In [ ]:
import json

from pyspark.sql import functions as F

from amsterdam_rental.plotting import render_potential_map

gold = spark.read.format("delta").load(f"{cfg.layer('gold')}/revenue_per_postcode")
rows = gold.select(
    "pc4",
    "investment_potential",
    "potential_score",
    "best_monthly_eur",
    "airbnb_avg_monthly_eur",
    "rental_avg_monthly_eur",
    "total_listing_count",
    "recommendation",
).collect()  # one row per postcode (~80), safe to collect

with open(cfg.source("post_codes.geojson")) as f:  # staged by 01_bronze
    features = json.load(f)["features"]

## Headline numbers

In [ ]:
both_sides = [
    r for r in rows if r.airbnb_avg_monthly_eur is not None and r.rental_avg_monthly_eur is not None
]
airbnb_wins = sum(r.recommendation == "airbnb" for r in both_sides)
print(f"Postcodes with data: {len(rows)}")
print(f"Airbnb out-earns long-term rent in {airbnb_wins} of {len(both_sides)} postcodes with both")

# Table view of the map: one row per tier
(
    gold.groupBy("potential_score", "investment_potential")
    .agg(
        F.count("*").alias("postcodes"),
        F.round(F.min("best_monthly_eur")).alias("min_best_monthly_eur"),
        F.round(F.max("best_monthly_eur")).alias("max_best_monthly_eur"),
    )
    .orderBy(F.col("potential_score").desc())
    .show(truncate=False)
)

## Map

In [ ]:
if cfg.on_databricks:
    MAP_PATH = f"{cfg.layer('figures')}/investment_potential_map.png"
else:
    MAP_PATH = os.path.join(REPO_ROOT, "docs", "images", "investment_potential_map.png")

counts = render_potential_map(
    features,
    scores={r.pc4: r.potential_score for r in rows},
    out_path=MAP_PATH,
    subtitle=(
        "Thirds of postcodes by highest achievable monthly income (Airbnb at 66% occupancy, "
        "or long-term rent).\n"
        f"Airbnb out-earns long-term rent in {airbnb_wins} of {len(both_sides)} postcodes "
        "with both. No sale prices available, so income is the proxy."
    ),
)
print(f"Map written to {MAP_PATH}")
print({k: v for k, v in counts.items()})

In [ ]:
# Show the PNG inline
from IPython.display import Image, display

display(Image(filename=MAP_PATH))